# Parte 2: Criação de Embeddings e Armazenamento (Vector Database)

Nesta etapa, completamos os pontos exigidos de:
- **Criação dos Embeddings**
- **Armazenamento e Indexação das Informações**

Vamos usar o modelo do Google Gemini ou um de código aberto local, e o ChromaDB para armazenar as representações matemáticas do texto.

In [ ]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import os
from tqdm import tqdm

DATA_DIR = '../data/'
PROCESSED_DIR = '../data/processed/'
DB_PATH = '../data/vector_db/'

# Carregando o CSV exportado pelo notebook 01
df_docs = pd.read_csv(f'{PROCESSED_DIR}reviews_enriched_for_rag.csv')
print(f"Total de documentos para indexação: {len(df_docs)}")

## 1. Instanciando o ChromaDB e os Embeddings

O ChromaDB é uma poderosa base de dados vetorial (Vector Store). 
**Estratégia**: Vamos armazenar localmente (`PersistentClient`). Para os embeddings, podemos utilizar a API do Google, ou usar a função default do Chroma (que usa SentenceTransformers locais como `all-MiniLM-L6-v2`) caso queiramos rodar 100% offline para não gerar custos no dataset de 40.000 reviews.

In [ ]:
# Inicia o ChromaDB com persistência de dados local
client = chromadb.PersistentClient(path=DB_PATH)

# Neste desafio, usaremos o embedding default (MiniLM) para não onerar o usuário com requisições de API em 40 mil itens.
# O MiniLM roda na CPU e indexa rapidamente o texto gerando um vetor denso.
sentence_transformer_ef = embedding_functions.DefaultEmbeddingFunction()

# Cria a coleção (tabela)
collection_name = "olist_reviews"
# Pega ou cria a coleção
collection = client.get_or_create_collection(
    name=collection_name, 
    embedding_function=sentence_transformer_ef,
    metadata={"hnsw:space": "cosine"} # Utilizamos distância de cosseno como métrica de indexação
)

## 2. Inserção (Indexação) em Lotes
Processar dezenas de milhares de avaliações em um comando só pode sobrecarregar a memória. 
**Estratégia**: Iremos criar `batches` de 1000 documentos e indexá-los acompanhando os IDs (necessários para recuperação de fontes na geração da resposta) e metadados (como a nota, caso queiramos filtrar depois).

In [ ]:
batch_size = 1000
total_docs = len(df_docs)

# Se a base já tiver os documentos, evitamos reinserir para o estudo não demorar muito ao rodar de novo.
if collection.count() < total_docs:
    print("Iniciando a indexação dos documentos...")
    for i in tqdm(range(0, total_docs, batch_size)):
        batch = df_docs.iloc[i:i+batch_size]
        
        # Prepara as listas para inserção no VectorDB
        documents = batch['rag_document'].tolist()
        ids = batch['review_id'].tolist()
        
        # Preparando os metadados (para permitirmos buscas exatas se necessário)
        metadatas = [{"review_score": int(score)} for score in batch['review_score'].tolist()]
        
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
    print("\nIndexação finalizada com sucesso!")
else:
    print(f"O banco já possui {collection.count()} registros indexados. Pulando indexação...")

## 3. Teste de Recuperação (Retrieval) com Filtragem de Metadados
**Solução de Problemas:** Se buscarmos por "Atraso na entrega e suporte ruim" usando uma métrica puramente geométrica (Cosine), o modelo pode acabar trazendo avaliações de nota 5 que dizem "Tudo ótimo, entrega rápida e bom suporte". Por quê? Porque as palavras "entrega" e "suporte" atraem a busca. 

**O Desafio Adicional:** Como previmos na documentação, precisamos usar **Metadados e Exploração Contextual**.
Para garantir que buscaremos apenas insatisfações relacionadas a atraso, adicionamos um filtro `where` exigindo que a nota da avaliação seja menor ou igual a 2.

In [ ]:
query_text = "Atraso na entrega e suporte ruim"
results = collection.query(
    query_texts=[query_text],
    n_results=3, # Retorna os 3 mais relevantes
    where={"review_score": {"$lte": 2}} # FILTRO DE METADADO: Nota menor ou igual a 2
)

print("Busca por:", query_text)
print("\n=== EVIDÊNCIAS RECUPERADAS (Com filtro de nota <= 2) ===")
for doc, id_, meta in zip(results['documents'][0], results['ids'][0], results['metadatas'][0]):
    print(f"ID: {id_}")
    print(f"Documento: {doc}")
    print("-"*50)